# Day 1 — RAG Fundamentals & RAGAS Core Metrics

**Module 5 · RAG Testing with RAGAS**

---

## What we'll cover today

| # | Topic | Why it matters |
|---|---|---|
| 1 | The retriever + generator flow | RAG fails in two independent stages — you need to know which one broke |
| 2 | RAGAS's 4 core metrics | `faithfulness`, `answer_relevancy` test generation; `context_precision`, `context_recall` test retrieval |
| 3 | Mapping metrics back to Module 3/4 | You already know most of these failure modes — RAGAS just measures them at a new layer |
| 4 | Setting up RAGAS | A judge LLM *and* an embeddings model this time |
| 5 | The annotated dataset schema | Same `category`/`failure_mode`/`is_hard_negative` habit from Module 4 Day 4 |

**Estimated time:** 60 minutes

---

> **Where we are in the course**
> Module 3 gave you the failure-mode vocabulary (hallucination, bias, toxicity...) and red-team threat categories (including RAG corpus poisoning).
> Module 4 gave you DeepEval's metrics, golden datasets, and — on Day 4 — the testing mindset itself: equivalence partitioning, boundary value analysis, the coverage matrix, hard negatives.
> None of that gets replaced today. RAGAS is a new library for a new *kind* of system (one with a retriever in front of the generator) — but the design process for deciding what to test is the one you already have.

---
## The retriever + generator flow

```
User question
    │
    ▼
RETRIEVER  →  searches a document store, returns the top-k most relevant chunks
    │
    ▼
GENERATOR  →  an LLM that writes an answer using the question + retrieved chunks
    │
    ▼
Final answer
```

Module 4's `FaithfulnessMetric` and `HallucinationMetric` test "does the answer stick to the provided context?" — but they treat the context as a given. They can't tell you whether the **right** context was retrieved in the first place. A RAG system fails in two different ways:

1. **The retriever fails** — wrong chunks, or the one chunk with the real answer is missing entirely.
2. **The generator fails** — the right chunk was retrieved, and the model still ignored or contradicted it.

> **Plain English:** if a RAG system gives a wrong answer, that's like getting a wrong answer from a research assistant. Did they grab the wrong book from the shelf (retriever failure), or did they have the right book open and still misread it (generator failure)? RAGAS is built to tell you which one happened.

---
## Real incident: Cursor's AI support agent invents a subscription policy (April 2025)

A user asked Cursor's AI-powered support chatbot why they'd been logged out after switching machines. The bot confidently explained that Cursor had introduced a new policy restricting users to a single device per subscription. That policy did not exist — the bot fabricated it. The user posted about canceling their subscription, the post went viral, other users assumed it was a real (unpopular) policy change, and Cursor's team had to publicly clarify that no such policy existed.

> **Why this matters for today:** this is the Air Canada incident from Module 4 Day 4, one layer removed. Air Canada's bot was *given* the wrong information and repeated it. Here, the support bot likely retrieved from a knowledge base that didn't contain a device-limit policy at all, or retrieved something tangentially related and misread it — a **retrieval-and-generation** failure. `faithfulness` and `context_precision` are the two metrics built specifically to catch this shape of bug.

---
## The 4 core RAGAS metrics

| Metric | Question it answers | Maps to (Module 3 / 4) |
|---|---|---|
| **`faithfulness`** | Does the answer only contain claims supported by the *retrieved* context? | Module 4 `FaithfulnessMetric`/`HallucinationMetric` — same idea, scoped to RAG |
| **`answer_relevancy`** | Does the answer actually address the question asked? | Module 4 `AnswerRelevancyMetric` |
| **`context_precision`** | Of the chunks retrieved, how many were actually relevant? | New — a **retrieval** failure mode |
| **`context_recall`** | Of all the relevant chunks that exist, how many did the retriever find? | New — requires a ground-truth reference |

`faithfulness` and `answer_relevancy` test the **generator**. `context_precision` and `context_recall` test the **retriever**. This split is what lets you localize a failure instead of just knowing "something is wrong."

> **Plain English:**
> - `context_precision` — of the books the research assistant pulled off the shelf, how many were actually useful?
> - `context_recall` — of every book in the library that could have answered the question, how many did they find?
> A research assistant can have perfect precision and terrible recall (or vice versa) — independent failure modes, measured independently.

---
## Setting up RAGAS

RAGAS needs **two** models configured: a judge LLM (for `faithfulness`/`answer_relevancy`) and an embeddings model (for the semantic-similarity scoring inside `context_precision`/`context_recall`). That's one more moving part than DeepEval, where the judge LLM alone was usually enough.

> **Note on this cell:** it requires `ragas` and the `openai` SDK installed (`pip install -r requirements.txt`) and either a local Ollama server or an `OPENAI_API_KEY`.
>
> For **Ollama** users, pull two models before running:
> ```
> ollama pull llama3.2:3b      # judge LLM (~2 GB)
> ollama pull all-minilm       # embeddings for answer_relevancy (~23 MB, smallest available)
> ```
> The setup cell uses `instructor.Mode.JSON_SCHEMA`, which makes Ollama enforce the output schema at the grammar level — without this, small local models echo the schema description back instead of filling it in.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import instructor
from openai import AsyncOpenAI
from ragas.llms import InstructorLLM
from ragas.embeddings import OpenAIEmbeddings

PROVIDER = os.getenv("PROVIDER", "ollama").lower()

if PROVIDER == "openai":
    raw_client = AsyncOpenAI()
    embed_client = AsyncOpenAI()
    llm_model   = os.getenv("DEMO_MODEL", "gpt-4o-mini")
    embed_model  = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
    instructor_client = instructor.from_openai(raw_client)
else:
    # Ollama: requires `ollama pull llama3.2:3b` and `ollama pull all-minilm`
    # Mode.JSON_SCHEMA lets Ollama enforce the output schema at the grammar level —
    # without it, llama3.2:3b echoes the schema description instead of filling it in.
    ollama_url  = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
    raw_client  = AsyncOpenAI(base_url=ollama_url, api_key="ollama")
    embed_client = AsyncOpenAI(base_url=ollama_url, api_key="ollama")
    llm_model   = os.getenv("DEMO_MODEL", "llama3.2:3b")
    embed_model  = os.getenv("OLLAMA_EMBED_MODEL", "all-minilm")   # separate from EMBEDDING_MODEL (OpenAI key)
    instructor_client = instructor.from_openai(raw_client, mode=instructor.Mode.JSON_SCHEMA)

judge_llm        = InstructorLLM(client=instructor_client, model=llm_model, provider="openai")
judge_embeddings = OpenAIEmbeddings(client=embed_client, model=embed_model)

print(f"Provider          : {PROVIDER}")
print(f"Judge LLM          : ready ({llm_model})")
print(f"Judge embeddings   : ready ({embed_model})")

/Users/takshinvarma/Desktop/AI-Testing-APR/module-05-ragas-rag-testing/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Provider          : ollama
Judge LLM          : ready (llama3.2:3b)
Judge embeddings   : ready (all-minilm)


---
## Building the dataset — reusing Module 4 Day 4's annotated schema

Module 4 Day 4 had you add `category`, `failure_mode`, and `is_hard_negative` to a golden dataset row. RAGAS renames the core fields (because retrieval is now under test too) but the annotation habit carries over unchanged:

| Module 4 field | RAGAS field | Why it changed |
|---|---|---|
| `input` | `user_input` | Same concept, RAGAS's naming |
| `actual_output` | `response` | Same concept |
| `context` | `retrieved_contexts` | Plural and explicit — these were *retrieved*, not handed to the model directly |
| `expected_output` | `reference` | Same concept — needed for `context_precision`/`context_recall` |
| `category`, `failure_mode`, `is_hard_negative` | *(unchanged)* | Still inert extra keys — still how you track coverage |

In [2]:
from ragas import EvaluationDataset

# Same annotation habit from Module 4 Day 4 — new core fields for the retrieval stage.
rows = [
    {
        "id": "policy-qa-01", "category": "policy_qa", "failure_mode": "hallucination", "is_hard_negative": False,
        "user_input": "What is our subscription's device limit policy?",
        "response": "There is no device limit — you can use your subscription on any number of devices.",
        "retrieved_contexts": [
            "Subscriptions are tied to an account, not a device. Users may log in from any device they own."
        ],
        "reference": "No device limit; the subscription is tied to the account, not a specific device.",
    },
    {
        "id": "policy-qa-01-hardneg", "category": "policy_qa", "failure_mode": "hallucination", "is_hard_negative": True,
        "user_input": "What is our subscription's device limit policy?",
        "response": "We limit each subscription to 1 device, with a 90-day grace period to switch devices.",
        "retrieved_contexts": [
            "Subscriptions are tied to an account, not a device. Users may log in from any device they own."
        ],
        "reference": "No device limit; the subscription is tied to the account, not a specific device.",
    },
]

dataset = EvaluationDataset.from_list(rows)
print(f"Dataset built with {len(dataset)} rows (1 normal case, 1 hard negative — the Cursor incident, reproduced)")

Dataset built with 2 rows (1 normal case, 1 hard negative — the Cursor incident, reproduced)


---
## Running the evaluation

> Live cell — needs the judge LLM/embeddings from the setup cell above, a real network call, and `await` (Jupyter supports top-level `await`). Expect this to take longer than a single DeepEval `assert_test()` call: `context_precision`/`context_recall` run several judge-LLM calls internally per row.
>
> **Small local models are a real limitation, not a hypothetical one:** RAGAS's judge prompts ask the model to decompose an answer into discrete statements and return structured JSON. A 3B model like `llama3.2:3b` frequently fails this — echoing the JSON schema back instead of filling it in, or wandering into unrelated text. If scores look wrong or the cell errors out, that's the judge model failing, not your RAGAS setup. Switch `PROVIDER=openai` with a real `OPENAI_API_KEY` for reliable judge output.

In [3]:
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

faithfulness      = Faithfulness(llm=judge_llm)
answer_relevancy  = AnswerRelevancy(llm=judge_llm, embeddings=judge_embeddings)
context_precision = ContextPrecision(llm=judge_llm)
context_recall    = ContextRecall(llm=judge_llm)

# Score from `rows` (not `dataset`) so the category/failure_mode/is_hard_negative
# annotations survive into the results table — EvaluationDataset only keeps the
# core RAGAS fields.
scored_rows = []
for row in rows:
    scored = dict(row)
    scored["faithfulness"] = (await faithfulness.ascore(
        user_input=row["user_input"], response=row["response"], retrieved_contexts=row["retrieved_contexts"],
    )).value
    scored["answer_relevancy"] = (await answer_relevancy.ascore(
        user_input=row["user_input"], response=row["response"],
    )).value
    scored["context_precision"] = (await context_precision.ascore(
        user_input=row["user_input"], reference=row["reference"], retrieved_contexts=row["retrieved_contexts"],
    )).value
    scored["context_recall"] = (await context_recall.ascore(
        user_input=row["user_input"], reference=row["reference"], retrieved_contexts=row["retrieved_contexts"],
    )).value
    scored_rows.append(scored)

import pandas as pd
df = pd.DataFrame(scored_rows)
print(df[["user_input", "category", "is_hard_negative",
          "faithfulness", "answer_relevancy", "context_precision", "context_recall"]].to_string())

# Expect: row 0 (normal) scores high on faithfulness; row 1 (hard negative) scores low —
# if it doesn't, this check isn't sensitive enough to catch the Cursor-shaped bug, same lesson as Module 4 Day 4.

                                        user_input   category  is_hard_negative  faithfulness  answer_relevancy  context_precision  context_recall
0  What is our subscription's device limit policy?  policy_qa             False      0.333333          0.884893                1.0             1.0
1  What is our subscription's device limit policy?  policy_qa              True      0.500000          0.819723                1.0             1.0


---
## Try It Yourself

1. Add a third row to `rows` above where the `retrieved_contexts` is **empty** (the retriever found nothing). What do you expect `faithfulness` and `answer_relevancy` to do when there's no context to be faithful to?
2. Add a fourth row that's a `context_precision` hard negative: `retrieved_contexts` should contain 3 chunks, only 1 of which is relevant to `user_input`. Predict the approximate `context_precision` score before running it.
3. Annotate both new rows with `category`, `failure_mode`, and `is_hard_negative`, exactly like Module 4 Day 4.

---
## Summary

### What we built today
- The retriever/generator failure split, and which RAGAS metric tests which side
- A real incident (Cursor) mapped onto that split
- A working RAGAS setup with a judge LLM and embeddings model
- A dataset using the same annotated schema from Module 4 Day 4, with RAGAS's renamed core fields
- A hard negative reproducing the Cursor incident, run through all 4 core metrics

### Carried forward unchanged from Module 4 Day 4
The annotation habit (`category`, `failure_mode`, `is_hard_negative`) and the instinct to build at least one hard negative per capability — RAGAS didn't need a new design process, just new field names.

**Next:** Day 2 — Chunking, Embeddings & Vector DB Validation, where we reproduce the chunk-boundary bug named (but not built) in Module 4 Day 4.

---